In [1]:

import sys
import ast
import numpy as np
import tabloo
import pickle
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
import pandas as pd
import os

# example command: 
# python   postprocess/step0_run_summary.py  /Volumes/data/Gilgamesh/kroppian/agovization_results/*

directories = ["/rdata/ian/pico/paperRuns/nsga2_baseline/nsga2_2000_2024-07-19_11-09-16"]

directories = [directory.rstrip('/') for directory in directories]

script_path = os.path.abspath('')

output_dir_seg = os.path.realpath(script_path).split("/")

output_dir = "/".join(output_dir_seg[0:-1])

IRR_COL = 1

raw_master_table = {'year':[], 'yield':[], 'leaching':[], 'irr_total':[], 'irr_app_count':[], 'front':[], 'scheds':[]}

raw_yrly_sum_tab = {
    "year" : [],
    "plant date" : [],
    "irr period start" : [],
    "irr period end": [],
    "nit period start" : [],
    "nit period end" : [],
    "max yield (kg/ha)": [],
    "mean I.A.C." : [],
    "median I.A.C." : [],
    "mean irr total (mm)" : [],
    "median irr total (mm)" : [],
    "mean leaching (kg/ha)" : [],
    "median leaching (kg/ha)" : [], 
    "min leaching (kg/ha)": [], 
    "max leaching (kg/ha)": []}


In [2]:
def parse_file_name(full_path, run_record):

    file_name = full_path.split("/")[-1]
    
    fields = file_name.split("_")

    run_record["algorithm"] = fields[0]
    run_record["year"] = int(fields[1])
    run_record["run_date"] = fields[2]
    run_record["run_time"] = fields[3]

    

In [3]:
def get_practice_info(run_record, year, genome_struct): 

    # Plant date 
    plant_date = 135

    if year % 4 == 0:
        plant_date += 1

    # Irr date info
    periods = genome_struct['date_ranges']
    irr_period = periods[0]
    irr_start = irr_period[0] - (year * 1e3)
    irr_end = irr_period[1] - (year * 1e3)

    # Nit date info
    nit_period = periods[1]
    nit_start = nit_period[0] - (year * 1e3)
    nit_end = nit_period[1] - (year * 1e3)

    # Yield information 
    max_yield = max(run_record['yield'])

    # Irrigation application counts and totals
    run_scheds = run_record['scheds'].tolist()
    #                               Only select the irr applications
    #                               V
    irr_app_count = [np.shape(sched[sched[:,IRR_COL] != 0])[0] for sched in run_scheds]
    #                          Only select the irr applications
    #                          V
    irr_app_total = [sum(sched[sched[:,IRR_COL] != 0][:,IRR_COL]) for sched in run_scheds]

    run_record['irr_app_count'] = irr_app_count
    run_record['irr_total'] = irr_app_total



In [4]:
def get_run_info(direcotry): 
    
    # Get the genome struct info 
    genome_struct_path = "%s/genome_structure.py" % directory

    with open(genome_struct_path, 'r') as f: genome_struct = ast.literal_eval(f.read())

    # Get the pickled run record
    run_record_path = "%s/run_sim_record.pkl" % directory
    infile = open(run_record_path, 'rb')
    run_record = pickle.load(infile)

    parse_file_name(direcotry, run_record)
    
    return (genome_struct, run_record)
    


    

In [5]:
def nds(run_record): 

    # Non-dominated sort
    f1 = -run_record['yield'] # Made this negative because we to maximize yield
    f2 = run_record['irr_total']
    F = np.column_stack((f1,f2))

    opt_fronts = NonDominatedSorting().do(F)

    record_count = run_record.shape[0]
    
    sorted_fronts = np.array([-1 for i in range(record_count)])

    for (f, front) in enumerate(opt_fronts):
        sorted_fronts[front] = f

    run_record['front'] = sorted_fronts.tolist()

    run_record = run_record[run_record['front'] == 0]

    return run_record
    

In [6]:
for directory in directories:

    (genome_struct, run_record) = get_run_info(directory)
    
    year = run_record["year"].iloc[0]

    print("Processing year %s for folder %s" % (year, directory))

    get_practice_info(run_record, year, genome_struct)

    run_record = nds(run_record)

    record_count = run_record.shape[0]
    
    # Build out the master table
    raw_master_table['year'] = raw_master_table['year'] + ([year] * record_count)

    for key in raw_master_table.keys():
        if key == 'year':
            continue
        raw_master_table[key] = raw_master_table[key] + run_record[key].tolist()


Processing year 2000 for folder /rdata/ian/pico/paperRuns/nsga2_baseline/nsga2_2000_2024-07-19_11-09-16


In [7]:
master_run_record = pd.DataFrame(raw_master_table)
yrly_sum_tab = pd.DataFrame(raw_yrly_sum_tab)

output = open('%s/master_run_record.pkl' % output_dir, 'wb')
pickle.dump(master_run_record, output)

output = open('%s/yearly_summary.pkl' % output_dir, 'wb')
pickle.dump(yrly_sum_tab, output)

# Creating XLSX output
print("Saving to excel...")

with pd.ExcelWriter('run_results.xlsx') as writer:  
    master_run_record.to_excel(writer, sheet_name='RunRecord')
    yrly_sum_tab.to_excel(writer, sheet_name='YearlySummary')


print("Done")

Saving to excel...
Done
